# Ejercicio 1. Detección de anomalías en el acceso al repositorio

Curso CIB-209, Temas Especiales en Seguridad de Datos y Sistemas.

Modificar únicamente la celda CONFIGURACIÓN. Ejecutar después todas las celdas en orden.

## Recordatorio de la técnica

k-means es aprendizaje automático no supervisado. No se le dice qué es normal ni cuáles eventos son incidentes: agrupa los accesos que se parecen entre sí y calcula el centro de cada grupo, que es el acceso promedio de ese grupo. Aquí cada grupo se llama perfil de comportamiento.

Sobre ese agrupamiento se mide la distancia de cada acceso al centro de su propio perfil. Mientras más lejos está, menos se parece a los demás accesos de su perfil. El percentil de corte define a partir de qué distancia se considera anomalía: con percentil 95 se marca el 5 por ciento de accesos más lejanos.

La columna clasificacion_forense no participa en el agrupamiento. Solo se usa al final para contar cuántos de los seis incidentes ya confirmados quedaron dentro de lo marcado.

## Vocabulario

- Perfil: grupo de accesos parecidos que k-means construyó solo, sin etiquetas.
- Centro del perfil: valor promedio del grupo en los cinco atributos.
- Distancia al centro: qué tan distinto es un acceso respecto de su propio grupo.
- Escalado de atributos: los cinco atributos se ponen en la misma escala antes de agrupar, para que el volumen en MB no pese más que la hora solo por tener números grandes.
- Anomalía: evento marcado por la distancia. No es lo mismo que incidente.

Este recordatorio explica la técnica y cómo leer las salidas. No interpreta los resultados, eso le corresponde al grupo.

In [ ]:
# ======================= CONFIGURACION =======================
NUMERO_DE_GRUPO = "G00"

numero_de_perfiles = 3      # corridas solicitadas: 3, 5, 5
percentil_de_corte = 95     # corridas solicitadas: 95, 95, 99
# =============================================================

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def sello_de_corrida(**parametros):
    texto = "|".join(f"{k}={parametros[k]}" for k in sorted(parametros))
    return hashlib.md5(texto.encode()).hexdigest()[:4].upper()

ATRIBUTOS = ["hora", "volumen_mb", "carpetas_distintas", "fuera_de_horario", "pais_inusual"]
eventos = pd.read_csv("eventos_acceso.csv")

print("Grupo:", NUMERO_DE_GRUPO)
print("numero_de_perfiles =", numero_de_perfiles, "  percentil_de_corte =", percentil_de_corte)
print("Sello de la corrida:", sello_de_corrida(perfiles=numero_de_perfiles, percentil=percentil_de_corte))
print()
print("Eventos cargados:", len(eventos))
print("Áreas:", ", ".join(sorted(eventos.area.unique())))
print(eventos.clasificacion_forense.value_counts().to_string())

In [ ]:
datos = StandardScaler().fit_transform(eventos[ATRIBUTOS])
modelo = KMeans(n_clusters=numero_de_perfiles, n_init=10, random_state=0).fit(datos)
distancia = np.linalg.norm(datos - modelo.cluster_centers_[modelo.labels_], axis=1)

eventos["perfil"] = modelo.labels_
eventos["distancia_al_centro"] = distancia.round(3)
corte = np.percentile(distancia, percentil_de_corte)
eventos["anomalia"] = (distancia > corte).astype(int)

perfiles = eventos.groupby("perfil").agg(
    eventos_en_el_perfil=("id_evento", "count"),
    hora_promedio=("hora", "mean"),
    volumen_promedio_mb=("volumen_mb", "mean"),
    carpetas_promedio=("carpetas_distintas", "mean"),
    area_predominante=("area", lambda s: s.value_counts().index[0])).round(1)
print("PERFILES DE COMPORTAMIENTO CONSTRUIDOS")
print(perfiles.to_string())
print()
print("Tamaño del perfil más pequeño:", int(perfiles.eventos_en_el_perfil.min()))
print("Distancia de corte:", round(float(corte), 3))

In [ ]:
marcadas = eventos[eventos.anomalia == 1]
MINUTOS_DE_TRIAJE_POR_ALERTA = 6

resumen = pd.DataFrame([
    ["Anomalías marcadas en total", len(marcadas)],
    ["Incidentes confirmados capturados (de 6)", int((marcadas.clasificacion_forense == "incidente_confirmado").sum())],
    ["Eventos de la campaña de respaldo marcados (de 30)", int((marcadas.clasificacion_forense == "respaldo_programado").sum())],
    ["Eventos normales marcados", int((marcadas.clasificacion_forense == "normal").sum())],
    ["Minutos de triaje requeridos", len(marcadas) * MINUTOS_DE_TRIAJE_POR_ALERTA],
], columns=["indicador", "valor"])
print("RESULTADO DE LA CORRIDA")
print(resumen.to_string(index=False))
print()
print("Anomalías marcadas por área")
print(marcadas.area.value_counts().to_string())

In [ ]:
print("DIEZ ANOMALÍAS CON MAYOR DISTANCIA AL CENTRO DE SU PERFIL")
columnas = ["id_evento", "usuario", "area", "hora", "volumen_mb", "carpetas_distintas",
            "pais", "distancia_al_centro", "clasificacion_forense"]
print(marcadas.sort_values("distancia_al_centro", ascending=False).head(10)[columnas].to_string(index=False))

## Cómo se lee el gráfico de dispersión

Un gráfico de dispersión coloca un punto por cada registro, ubicado según dos de sus atributos. Aquí cada punto es un acceso al repositorio: el eje horizontal es la hora del día en que ocurrió, de 0 a 23, y el eje vertical es el volumen descargado en MB. Un punto arriba y a la izquierda es una descarga grande en la madrugada.

Los cuatro símbolos son:

- Punto gris: acceso que la corrida no marcó.
- Punto naranja: acceso marcado como anomalía.
- Círculo rojo sin relleno: uno de los seis incidentes confirmados por el equipo forense.
- Cuadro azul sin relleno: uno de los 30 eventos de la campaña de respaldo programada.

Para leerlo conviene revisar tres cosas: cuántos círculos rojos quedaron pintados de naranja, cuántos puntos naranja caen sobre cuadros azules, y cuántos puntos naranja están dentro de la nube gris, es decir, junto a accesos que nadie considera sospechosos.

Advertencia importante: el modelo trabaja con cinco atributos y el gráfico muestra solo dos. Un punto puede estar marcado por lo que ocurre en los otros tres, por ejemplo el país inusual, y por eso a veces se ve un punto naranja en medio de puntos grises.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
normales = eventos[eventos.anomalia == 0]
ax.scatter(normales.hora, normales.volumen_mb, s=12, c="#b0b7c3", label="no marcado")
ax.scatter(marcadas.hora, marcadas.volumen_mb, s=26, c="#d97706", label="marcado como anomalía")
inc = eventos[eventos.clasificacion_forense == "incidente_confirmado"]
ax.scatter(inc.hora, inc.volumen_mb, s=120, facecolors="none", edgecolors="#b91c1c",
           linewidths=1.8, label="incidente confirmado")
res = eventos[eventos.clasificacion_forense == "respaldo_programado"]
ax.scatter(res.hora, res.volumen_mb, s=60, marker="s", facecolors="none",
           edgecolors="#1d4ed8", linewidths=1.2, label="campaña de respaldo")
ax.set_xlabel("hora del acceso")
ax.set_ylabel("volumen descargado en MB")
ax.set_title(f"Grupo {NUMERO_DE_GRUPO} | numero_de_perfiles={numero_de_perfiles} | percentil_de_corte={percentil_de_corte} | sello={sello_de_corrida(perfiles=numero_de_perfiles, percentil=percentil_de_corte)}")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()